# ToolRetryMiddleware
## 重试策略。

基于指数退避算法，设置工具调用失败时的重试策略。

指数退避（Exponential Backoff）的核心思想就是：当某个操作失败（通常是网络请求、API调用或数据库连接）时，系统不会立刻重试，也不会每次都等待相同的固定时间，而是让每一次重试的延迟时间按指数级增长。

## 为什么不直接重试？
想象一下，某个热门网站的服务器因为瞬间流量太大（比如抢票或秒杀）崩溃了。如果所有失败的客户端都立刻或每隔1秒就重试一次，这无异于对已经瘫痪的服务器进行了一场持续的DDoS（分布式拒绝服务）攻击，服务器可能永远也缓不过来。

jitter是为了避免大量工具的重试请求集中在固定的时间点，引入抖动。
假设：按照策略，两次工具调用请求的时间间隔应为10秒，加入抖动后，可能为8.9秒，也可能为10.2秒。

# 举例1：带抖动的重试策略

In [2]:

from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware
from langchain.messages import HumanMessage
import datetime
from langchain.tools import tool

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")
)

def write_times(s):
    """将每次工具调用的时间戳和间隔写入本地文件，方便观察退避策略"""
    with open("call_times_with_jitter.txt", "a", encoding="utf-8") as f:
        f.write(s + "\n")


count = 1
start_time = None


@tool
def get_weather(city: str):
    """查询指定城市天气"""
    global count
    global start_time
    interval = 0
    current_time = datetime.datetime.now()
    if not start_time:
        interval = 0
    else:
        # 计算当前调用与上一次调用之间的时间差（秒）
        interval = (current_time - start_time).total_seconds()
    start_time = current_time
    res_str = f"第 {count} 次调用，当前时间：{start_time}，和上次调用间隔 {interval} 秒"
    count += 1
    # 记录日志
    write_times(res_str)
    # 故意抛出 TimeoutError，以此触发中间件的重试机制
    raise TimeoutError("Not Implemented")


agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[
        # ToolRetryMiddleware 用于捕获工具执行中的异常并自动重试
        ToolRetryMiddleware(
            max_retries=6,  # 最大重试次数（不包含初始的那次调用，一共最多调 1 + 6 = 7 次）
            backoff_factor=2.0,  # 指数退避因子（每次重试等待时间乘以 2）
            initial_delay=1.0,  # 第一次重试前的初始等待时间（1 秒）
            max_delay=10.0,  # 最大等待延迟上限（防止指数增长无限大，限制在 10 秒）
            jitter=True,  # 开启抖动（在等待时间中加入随机性，防止并发请求时出现“惊群效应”）
            retry_on=(TimeoutError,),  # 仅针对捕获到特定的 TimeoutError 异常时才触发重试
            on_failure="continue"  # 当达到最大重试次数依然失败时，Agent 的行为为："continue" 表示将错误信息包装后塞回对话历史，让大模型知道失败了并继续决策
        ),
    ],
)

response = agent.invoke({
    "messages": [HumanMessage("今天北京天气如何？")]
})

# 1. 你的提问 -> 2. AI 决定调用工具 -> 3. 重试失败后的错误反馈 -> 4. AI 最终给出的兜底回复
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

今天北京天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_YdCvBGkyQl0XQWOTiadEVX8T)
 Call ID: call_YdCvBGkyQl0XQWOTiadEVX8T
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

Tool 'get_weather' failed after 7 attempts with TimeoutError: Not Implemented. Please try again.
================================== Ai Message ==================================

抱歉，我目前无法获取北京的实时天气数据（天气查询接口暂时不可用）。

如果你愿意，我可以：
1. 帮你整理一个“北京天气查询”的快速方法；
2. 根据你提供的天气截图/数据，帮你解读；
3. 先告诉你北京此时通常该怎么穿衣、带伞建议。


# 举例2：无抖动

自动重试失败的工具调用。

第1次重试(retry_number=1):等待～1.0*(2.0**1)=2.0秒

第2次重试(retry_number=2):等待～1.0*(2.0**2) = 4.0秒

第3次重试(retry_number=3):等待～1.0*(2.0**3) =8.0秒

也就是说，等待时间以指数方式增长一一每失败一次，下次再试之前等待更长时间。

如果你把backoff_factor=0，就意味着不使用指数增长，重试之间始终用固定的initial_delay。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware
from langchain.messages import HumanMessage
import datetime


def write_times(s):
    """将每次工具调用的时间戳和间隔写入本地文件，方便观察退避策略"""
    with open("call_times_with_jitter.txt", "a", encoding="utf-8") as f:
        f.write(s + "\n")


count = 1
start_time = None


@tool
def get_weather(city: str):
    """查询指定城市天气"""
    global count
    global start_time
    interval = 0
    current_time = datetime.datetime.now()
    if not start_time:
        interval = 0
    else:
        # 计算当前调用与上一次调用之间的时间差（秒）
        interval = (current_time - start_time).total_seconds()
    start_time = current_time
    res_str = f"第 {count} 次调用，当前时间: {start_time}，和上次调用间隔 {interval} 秒"
    count += 1
    # 记录日志
    write_times(res_str)
    # 故意抛出 TimeoutError，以此触发中间件的重试机制
    raise TimeoutError("Not Implemented")


agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[
        # ToolRetryMiddleware 用于捕获工具执行中的异常并自动重试
        ToolRetryMiddleware(
            max_retries=6,  # 最大重试次数（不包含初始的那次调用，一共最多调 1 + 6 = 7 次）
            backoff_factor=2.0,  # 指数退避因子（每次重试等待时间乘以 2）
            initial_delay=1.0,  # 第一次重试前的初始等待时间（1 秒）
            max_delay=10.0,  # 最大等待延迟上限（防止指数增长无限大，限制在 10 秒）
            jitter=False,  # 关闭抖动，意味着重试机制从"随机化的指数退避"退化成了"严格固定的指数退避"
            retry_on=(TimeoutError,),  # 仅针对捕获到特定的 TimeoutError 异常时才触发重试
            on_failure="continue"  # 当达到最大重试次数依然失败时，Agent 的行为："continue" 表示将错误信息包装后塞回对话历史，让大模型知道失败了并继续决策
        ),
    ],
)

response = agent.invoke({
    "messages": [HumanMessage("今天北京天气如何？")]
})

# 1. 你的提问 -> 2. AI 决定调用工具 -> 3. 重试失败后的错误反馈 -> 4. AI 最终给出的兜底回复
for msg in response["messages"]:


# 对比

将 jitter 从 True 改为 False（关闭抖动），意味着重试机制从“随机化的指数退避”退化成了“严格固定的指数退避”。

为了更直观理解，看一下这两种状态下的核心区别：
## 1. 理论上的等待时间对比
在这段代码中，你设置了 `initial_delay=1.0`（初始延迟 1 秒）、`backoff_factor=2.0`（倍数是 2）以及 `max_delay=10.0`（最大延迟 10 秒）。
当工具持续报错时，关闭抖动（`jitter=False`）与开启抖动（`jitter=True`）的等待延迟（Interval）对比如下：

| 重试轮次 | 理想基础延迟 (秒) | 关闭抖动 (jitter=False) 的实际等待 | 开启抖动 (jitter=True) 的实际等待 |
| ---- | ---- | ---- | ---- |
| 第1次重试 | $1.0 × 2^0 = 1.0$ | 严格等于 1.0 秒 | 在 0 ~ 1.0 秒之间随机 |
| 第2次重试 | $1.0 × 2^1 = 2.0$ | 严格等于 2.0 秒 | 在 0 ~ 2.0 秒之间随机 |
| 第3次重试 | $1.0 × 2^2 = 4.0$ | 严格等于 4.0 秒 | 在 0 ~ 4.0 秒之间随机 |
| 第4次重试 | $1.0 × 2^3 = 8.0$ | 严格等于 8.0 秒 | 在 0 ~ 8.0 秒之间随机 |
| 第5次重试 | $1.0 × 2^4 = 16.0 → 10.0$ | 严格等于 10.0 秒（受制于 max_delay） | 在 0 ~ 10.0 秒之间随机 |
| 第6次重试 | $1.0 × 2^5 = 32.0 → 10.0$ | 严格等于 10.0 秒（受制于 max_delay） | 在 0 ~ 10.0 秒之间随机 |
💡 现象结论：
关闭抖动后，查看生成的 `call_times_with_jitter.txt` 日志，你会发现输出的 interval 数字会极其精准地趋近于 1.0、2.0、4.0、8.0、10.0、10.0。

---
## 2. 为什么要引入 Jitter（抖动）？关闭它会有什么问题？
在单用户、单并发的测试环境下，关闭 jitter 没有任何副作用，甚至能让等待时间非常规律、可预测。

但在高并发的生产环境中，关闭 jitter 会引发灾难性的 **惊群效应（Thundering Herd Problem）**：
- 没有 Jitter 的惨剧（`jitter=False`）：
假设某天气 API 服务突然宕机了 1 秒。此时刚好有 1000 个用户同时发起了查询。因为这 1000 个请求同时失败，并且它们都严格死板地等待 1 秒、2 秒、4 秒……
这意味着，在第 1 秒、第 3 秒、第 7 秒的那个精准时间点上，这 1000 个请求会整整齐齐地再次轰炸服务器。刚刚复活的服务器瞬间又被这波峰值流量压垮，形成恶性循环。

- 引入 Jitter 的优势（`jitter=True`）：
通过给重试时间加上随机性，这 1000 个请求会在 0～1 秒、0～2 秒的区间内均匀地错开（削峰填谷）。流量被平摊到了整条时间轴上，服务器就能轻松地分批处理完这些请求。

## 总结：
- `jitter=False`（你当前的代码）：重试间隔死板、精准、可预测。适合本地调试、测试重试逻辑是否生效。
- `jitter=True`：重试间隔随机、错开、更安全。适合线上生产环境，防止把下游第三方 API 或数据库冲垮。
